# 🧾 Invoice Agent — Colab Launcher
### Pulls the latest code from GitHub and runs the full Streamlit UI via a public URL

---

## Quick-start (3 steps)

1. **Set your GitHub repo URL** in Cell 1 below
2. **Add your API key** to Colab Secrets (🔑 left sidebar)
   - `GEMINI_API_KEY` — free key from [aistudio.google.com](https://aistudio.google.com/app/apikey) *(recommended)*
   - *or* `ANTHROPIC_API_KEY` — if you prefer Claude
3. **Runtime → Run all**

> **Login credentials:** `demo` / `demo`

---

## Step 1 — Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS before running                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"
BRANCH      = "main"          # branch to pull
PROJECT_DIR = "/content/invoice-agent"

# VLM backend: "gemini" (free, recommended) | "claude" (paid)
VLM_BACKEND = "gemini"

print(f"Repo  : {GITHUB_REPO}")
print(f"Branch: {BRANCH}")
print(f"VLM   : {VLM_BACKEND}")

## Step 2 — Clone / Pull Latest Code

In [ ]:
import os

if os.path.exists(PROJECT_DIR):
    print("📂 Repo already present — pulling latest changes...")
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} reset --hard origin/{BRANCH}
    !git -C {PROJECT_DIR} pull origin {BRANCH}
else:
    print("📥 Cloning repository...")
    !git clone --branch {BRANCH} {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print("\n✅ Code is up to date. Last 5 commits:")
!git log --oneline -5

## Step 3 — Install System Dependencies

In [ ]:
%%capture cap
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils libgl1 libglib2.0-0
print("✅ System dependencies installed (tesseract, poppler, libgl1).")

## Step 4 — Install Python Packages

In [ ]:
%%capture cap
!pip install -q -r {PROJECT_DIR}/requirements.txt
# mlx-vlm is Apple Silicon only — skip on Colab
print("✅ Python packages installed.")

## Step 5 — Configure API Keys & Environment

In [ ]:
import os, getpass

def _secret(name):
    """Try Colab Secrets first, fall back to manual input."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            print(f"✅ {name} loaded from Colab Secrets.")
            return val
    except Exception:
        pass
    return getpass.getpass(f"Enter {name}: ")

env_lines = [
    f"VLM_BACKEND={VLM_BACKEND}",
    f"SQLITE_PATH={PROJECT_DIR}/data/invoices.db",
    f"CHROMA_PERSIST_DIR={PROJECT_DIR}/data/chroma",
]

if VLM_BACKEND == "gemini":
    gemini_key = _secret("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"] = gemini_key
    env_lines.append(f"GEMINI_API_KEY={gemini_key}")
    env_lines.append("GEMINI_MODEL=gemini-2.5-flash")
else:
    claude_key = _secret("ANTHROPIC_API_KEY")
    os.environ["ANTHROPIC_API_KEY"] = claude_key
    env_lines.append(f"ANTHROPIC_API_KEY={claude_key}")
    env_lines.append("CLAUDE_MODEL=claude-sonnet-4-6")

with open(f"{PROJECT_DIR}/.env", "w") as f:
    f.write("\n".join(env_lines) + "\n")

os.makedirs(f"{PROJECT_DIR}/data/samples", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data/output",  exist_ok=True)

print("✅ .env written. Data directories ready.")

## Step 6 — Launch Streamlit + Public URL

Starts the app on port 8501 and opens a Cloudflare Tunnel so anyone can access it.
The public URL is printed below — share it with your team.

> **Login:** `demo` / `demo`

In [ ]:
import subprocess, threading, time, re, sys, os

os.chdir(PROJECT_DIR)

# ── 1. Kill any stale Streamlit/cloudflared processes ────────────────────
!pkill -f streamlit    2>/dev/null || true
!pkill -f cloudflared  2>/dev/null || true
time.sleep(2)

# ── 2. Download cloudflared binary ───────────────────────────────────────
cf_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_bin):
    print("⬇️  Downloading cloudflared...")
    !wget -q -O {cf_bin} \
        https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {cf_bin}
    print("✅ cloudflared ready.")

# ── 3. Start Streamlit ───────────────────────────────────────────────────
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.fileWatcherType", "none",
    ],
    cwd=PROJECT_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("⏳ Waiting for Streamlit to start...")
time.sleep(5)

# ── 4. Open Cloudflare Tunnel ────────────────────────────────────────────
tunnel_proc = subprocess.Popen(
    [cf_bin, "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline().decode("utf-8", errors="ignore")
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group()
        break
    time.sleep(1)

if public_url:
    print(f"""
╔══════════════════════════════════════════════════════════════╗
║  🚀  Invoice Agent is LIVE                                   ║
║                                                              ║
║  URL   : {public_url:<48}║
║  Login : demo / demo                                         ║
║                                                              ║
║  Share this link — active while this session is running.     ║
╚══════════════════════════════════════════════════════════════╝
""")
else:
    print("❌ Cloudflare tunnel failed. Try the localtunnel fallback in the next cell.")

### Step 6b — Fallback: localtunnel (run only if Step 6 failed)

In [ ]:
# Only run this cell if Step 6 did NOT produce a URL
import urllib.request
ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
print(f"When asked for a password, enter: {ip}")
!npm install -g localtunnel --silent
!lt --port 8501

---
## Step 7 — Pull Latest Updates (re-run anytime)
Run this cell to pull new commits from GitHub without restarting the session.

In [ ]:
import subprocess, sys, time, os

print("⬇️  Pulling latest changes from GitHub...")
!git -C {PROJECT_DIR} fetch origin
!git -C {PROJECT_DIR} reset --hard origin/{BRANCH}
!git -C {PROJECT_DIR} pull origin {BRANCH}
print("\n✅ Code updated. Last 5 commits:")
!git -C {PROJECT_DIR} log --oneline -5

# Restart Streamlit to pick up changes
!pkill -f streamlit 2>/dev/null || true
time.sleep(2)
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.fileWatcherType", "none"],
    cwd=PROJECT_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(4)
print(f"\n🔄 Streamlit restarted — refresh the URL above.")

---
## Step 8 — Contribute Changes Back to GitHub
Edit files in `/content/invoice-agent/` via the Colab file browser, then commit here.

In [ ]:
# Run once per session to set your identity
GIT_NAME  = "Your Name"          # ← change this
GIT_EMAIL = "you@example.com"    # ← change this

!git config --global user.name  "{GIT_NAME}"
!git config --global user.email "{GIT_EMAIL}"
print("✅ Git identity configured.")

In [ ]:
COMMIT_MESSAGE = "Describe your changes here"   # ← change this

!git -C {PROJECT_DIR} status
!git -C {PROJECT_DIR} add -A
!git -C {PROJECT_DIR} commit -m "{COMMIT_MESSAGE}"
!git -C {PROJECT_DIR} push origin {BRANCH}

---
## Reference

| Task | What to do |
|---|---|
| Open the app | Use the URL printed in Step 6 |
| Process an invoice | Login → **Process Document** → upload PDF/image |
| Review extractions | Login → **Review Queue** |
| Check dashboard metrics | Login → **Dashboard** |
| Pull new code | Re-run **Step 7** |
| Push your edits | Fill in Step 8 and run |
| Fresh environment | Runtime → Factory reset, then Run all |

### Project layout
```
invoice-agent/
├── app.py                  ← Streamlit entry point
├── config.py               ← All settings & env vars
├── ui/pages/               ← Dashboard, Process, Review, History …
├── agents/                 ← Extraction, validation, correction agents
├── graph/workflow.py       ← LangGraph pipeline state machine
├── models/schemas.py       ← Pydantic data models
├── database/storage.py     ← SQLite CRUD + aggregate stats
└── requirements.txt
```

### Supported VLM backends
| Backend | Key needed | Cost |
|---|---|---|
| `gemini` | `GEMINI_API_KEY` | Free tier |
| `claude` | `ANTHROPIC_API_KEY` | Paid |
